# K1足球坐标：工训课单目几何测试

检测框底部取点 → 相机射线 → 地面求交。OS SDK取数与检测，自编函数计算像素到米。静态只读测试，不发送动作。py与本Notebook放在同一目录。

## 1. 填写实测镜头高度

机器人、头部与球保持静止，地面平整，机身直立。量镜头中心到地面的**垂直高度**，单位米。90 cm整机身高不能直接替代此数。当前默认目标帧为trunk；程序会检查本次帧列表。无需教师JSON。

In [ ]:
from geometry_ball_coordinates import run_geometry_probe

CAMERA_HEIGHT_M = None  # 填入当前姿态下实测的镜头中心离地高度（m）
KNOWN_XY = None  # 可选：相同身体原点/轴向的实测(x,y)，仅用于比较
NOTE = "静止测试；补充卷尺起点、球上终点和摆放位置"


## 2. 采一张新图并计算

从SDK读取RGB、内参和optical→trunk TF，再执行soccer模型检测。选取最高置信度Ball，沿用工训课框底向上10%的像素规则。保持静止直至打印结果。

In [ ]:
report = run_geometry_probe(
    camera_height_m=CAMERA_HEIGHT_M,
    known_xy=KNOWN_XY,
    note=NOTE,
)


## 3. 检查结果

成功状态为geometry_candidate；x向前为正、y向左为正，单位m。point_base_m的z约为0；point_body_m的z可能为负。ray_length_m是眼睛到求交点的斜距，distance_xy_m是地面平面距离。标注图红点是求交像素。候选值仍需现场核对，不直接启动踢球。

In [ ]:
from pathlib import Path
from IPython.display import display, Image

print("状态：", report["status"])
print("取点像素：", report.get("pixel_uv"))
print("姿态：", report.get("pose"))
print("几何：", report.get("geometry"))
print("踢球参数预览：", report["kick_input_preview"])
print("异常：", report.get("error"))
print("记录位置：", report["output_dir"])
image_path = Path(report["output_dir"]) / "selected_ball.png"
if image_path.exists():
    display(Image(filename=str(image_path)))


## 4. 距离对照与保存

核对同一张图的足球与红点；量地面距离时记录起点是躯干原点投影、脚尖还是其他位置，终点是球前缘还是接地点。原点未统一时先保留量法说明，不自动算误差。需要对照其他球位时可重新摆球、保持静止，再执行第2步。每次自动保存独立文件夹和同名结果ZIP。

## 5. 缺少trunk或其他异常

若list_frames只有五个头部帧，程序会记录缺少身体姿态，坐标为空。头部帧间TF不能代替头部到身体的姿态。旧工训课使用/head_pose提供这项信息；若已有可信旧位姿，可使用py的head_to_ground入口，说明见随包Markdown。不要为得到数字改填头部目标帧或单位矩阵。

详细来源、公式及原点说明见《单目几何测试说明.md》。旧现场数据及深度回读继续保留，本Notebook尚未执行真机。